# 11j — Weekly identifiability of contactee mean vs neighbourhood-mean degree

The per-week contact model (`constant_contacts=false`) fits a **hierarchical spatio-temporal GP**: a per-week level `c_t`, a matrix-normal structure field, and a **per-week dispersion random effect `τ_t`** (the *variance term*). This notebook reconstructs, read-only from the cached `8j_s1_*` Stage-1 chains (no refit, no Stage 2), the weekly **mean** degree `⟨k⟩` (`MeanNGM` C0) and **neighbourhood-mean** degree `⟨k²⟩/⟨k⟩·g` (`NeighbourhoodDegreeNGM` C0), with **90% CIs** across the Stage-1 draws, for a **contactee** of age **25-34** (bin 4) and **70+** (bin 7).

One panel per **contactor** age *i* (7 panels), for each degree model (`unweighted-negbin`, `weighted-hweibull`). The mean depends only on the level; the neighbourhood mean is driven by the second moment / `τ_t`. So **wider ribbons or larger week-to-week jumps in the neighbourhood line than the mean line** are the signature of the per-week variance term — and show which weeks are (un)identifiable.

In [ ]:
ENV["GKSwstype"] = "100"   # headless GR (off-screen PNG) for nbconvert
include("forecast_utils.jl")   # single preamble: base + CoMix pipeline + forecasting framework
using Random, Statistics
mkpath("../res")

default_plot_setting()

In [ ]:
cfg  = FrameworkConfig(constant_contacts = false)   # per-week GP — required for weekly fluctuation
grid = cis_age_grid()
@assert grid.LAB[4] == "25-34" && grid.LAB[7] == "70+"
@assert contacts_label(cfg) == "temporal-gsar-cut-sc-hd-p0-gi"

raw  = load_raw_contact_inputs()
FORECAST_ORIGINS = available_forecast_origins(cfg; grid = grid, craw = raw.craw)

ORIGIN = Date(2021, 5, 9)
@assert ORIGIN in FORECAST_ORIGINS "ORIGIN $(ORIGIN) not in $(first(FORECAST_ORIGINS))…$(last(FORECAST_ORIGINS))"
win = WeeklyWindow(ORIGIN; n_fit = cfg.n_fit, smax = cfg.smax, horizons = cfg.horizons)
wd  = load_window_data(win; grid = grid)

println("origin        : ", ORIGIN)
println("contacts token: ", contacts_label(cfg))
println("contactees    : ", grid.LAB[4], " (j=4), ", grid.LAB[7], " (j=7)")

In [ ]:
models = [NegBinAgePair(), HurdleWeibullAgePair()]
include("8j_viz_utils.jl")    # stage1_chain_path
include("11j_viz_utils.jl")   # moment_timeline_stats, plot_moment_timeline
CONTACTEES = (4, 7)           # 25-34, 70+
println("degree models : ", degree_label.(models))

## Weekly mean vs neighbourhood-mean degree (90% CI)

Four figures = 2 contactees × {`unweighted-negbin`, `weighted-hweibull`}. Each is a 7-panel grid (contactor age *i*); steelblue = mean `⟨k⟩`, darkorange = neighbourhood `⟨k²⟩/⟨k⟩`, ribbons = 90% CI, dashed line = forecast origin. Faint gray bars (secondary right axis) = the per-cell sample size `n_pos = n_roster·(1−p⁰)` (positive contacts in that cell that week) — read the CI width against it: a wide ribbon over thin bars is a sample-starved, poorly-identified week. Saved to `../res/11j_moment_timeline_*`.

In [ ]:
# unweighted-negbin — contactee 25-34 then 70+
for j in CONTACTEES
    display(plot_moment_timeline(NegBinAgePair(), j, ORIGIN, cfg, grid, raw, wd))
end

In [ ]:
# weighted-hweibull — contactee 25-34 then 70+ (K1 = (1−p⁰)·μW; neighbourhood carries the variance term)
for j in CONTACTEES
    display(plot_moment_timeline(HurdleWeibullAgePair(), j, ORIGIN, cfg, grid, raw, wd))
end